## import thu vien

In [ ]:
from typing import List, Dict, Any
from langchain_core.runnables.history import RunnableWithMessageHistory
from prompts.prompt import contextualize_q_system_prompt, qa_system_prompt
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_openai import ChatOpenAI
from service.func_for_fc import rag_doctor_info, rag_product_info, rag_service_info, book_appointment, qa_medical, qa_symptom
import os
import json
import time
import streamlit as st
from langchain.chains import create_history_aware_retriever, create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda
from langchain_openai import ChatOpenAI
# import uuid
import atexit
from service.message_stored import load_session_history, get_db, save_message
from datetime import datetime

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from dotenv import load_dotenv
import os
from service.search_doc import hybrid_search
from openai import OpenAI

load_dotenv('/mnt/data1tb/thangcn/datnv2/.env')

open_ai_key = os.getenv("OPENAI_API_KEY")
MODEL = 'gpt-4o' 
EMBED_MODEL = "thang1943/multilingual-e5-large-v2"

client = OpenAI(
    api_key = open_ai_key
)

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={'device': 'cpu'}
)


In [ ]:
store = {}
session_id = 'thangcn1'

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = load_session_history(session_id)
    return store[session_id]

def save_all_sessions():
    for session_id, chat_history in store.items():
        for message in chat_history.messages:
            save_message(session_id, message["role"], message["content"])

In [ ]:
query = "Nuoc yen sao"

In [ ]:
def merge_faiss_stores(embeddings, index_paths):
    if not index_paths:
        return None
    
    # Load index đầu tiên làm base
    merged_store = FAISS.load_local(
        index_paths[0], 
        embeddings, 
        allow_dangerous_deserialization=True
    )
    
    # Merge các index còn lại
    for path in index_paths[1:]:
        store = FAISS.load_local(
            path, 
            embeddings, 
            allow_dangerous_deserialization=True
        )
        merged_store.merge_from(store)
    
    return merged_store

# Sử dụng:
index_paths = [
    '/mnt/data1tb/thangcn/datnv2/vector_database/FAISS_2/service_info',
    '/mnt/data1tb/thangcn/datnv2/vector_database/FAISS_2/product_info',
    '/mnt/data1tb/thangcn/datnv2/vector_database/FAISS_2/doctor_info',
    '/mnt/data1tb/thangcn/datnv2/vector_database/FAISS_2/qa_document',
    '/mnt/data1tb/thangcn/datnv2/vector_database/FAISS_2/symptoms'
]

merged_vectorstore = merge_faiss_stores(embeddings, index_paths)

In [ ]:
ensemble_retriever = hybrid_search(merged_vectorstore, query, k=10)

OutOfMemoryError: CUDA out of memory. Tried to allocate 978.00 MiB. GPU 0 has a total capacity of 23.65 GiB of which 620.75 MiB is free. Process 445557 has 20.53 GiB memory in use. Including non-PyTorch memory, this process has 2.47 GiB memory in use. Of the allocated memory 2.09 GiB is allocated by PyTorch, and 12.18 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
llm = ChatOpenAI(
    base_url="http://localhost:8000/v1",
    api_key="dummy",
    temperature=0.01,
    model="thang1943/Llama-3.1-8B-instruction-final",
)

In [ ]:
current_datetime = datetime.now().strftime("%H:%M:%S %-m/%-d/%y")

In [ ]:
def process_llm_function_call(chat_history, user_prompt: str):
    messages = [{
        "role": "system",
        "content": f'MED|{current_datetime}|Your name is HCAI|Respond to medical queries only|For non-medical: "I only handle medical queries"'
    }]

    for msg in chat_history.messages[max(-len(chat_history.messages), -3):]:
        messages.append(msg)

    messages.append(
        {"role": "user", "content": user_prompt}
    )
    # Gọi LLM với function calling
    response = llm.predict_messages(
        messages,
    )
    print(response)
    return response

In [ ]:
def create_contextualize_prompt(contextualize_q_system_prompt, qa_system_prompt):
    contextualize_q_prompt = ChatPromptTemplate.from_messages(
        [
            ("system", contextualize_q_system_prompt), 
            MessagesPlaceholder("chat_history"), 
            ("human", "{input}")
        ]
    )

    qa_prompt = ChatPromptTemplate.from_messages(
        [("system", qa_system_prompt), MessagesPlaceholder("chat_history"), ("human", "{input}")]
    )

    return contextualize_q_prompt, qa_prompt


contextualize_q_prompt, qa_prompt = create_contextualize_prompt(contextualize_q_system_prompt, qa_system_prompt)